# Synthetic Data Generation Using RAGAS - RAG Evaluation with LangSmith

In the following notebook we'll explore a use-case for RAGAS' synthetic testset generation workflow!



- 🤝 BREAKOUT ROOM #1
  1. Use RAGAS to Generate Synthetic Data

- 🤝 BREAKOUT ROOM #2
  1. Load them into a LangSmith Dataset
  2. Evaluate our RAG chain against the synthetic test data
  3. Make changes to our pipeline
  4. Evaluate the modified pipeline

SDG is a critical piece of the puzzle, especially for early iteration! Without it, it would not be nearly as easy to get high quality early signal for our application's performance.

Let's dive in!

# 🤝 BREAKOUT ROOM #1

## Task 1: Dependencies and API Keys

We'll need to install a number of API keys and dependencies, since we'll be leveraging a number of great technologies for this pipeline!

1. OpenAI's endpoints to handle the Synthetic Data Generation
2. OpenAI's Endpoints for our RAG pipeline and LangSmith evaluation
3. QDrant as our vectorstore
4. LangSmith for our evaluation coordinator!

Let's install and provide all the required information below!

## Dependencies and API Keys:

### NLTK Import

To prevent errors that may occur based on OS - we'll import NLTK and download the needed packages to ensure correct handling of data.

In [1]:
import nltk
nltk.download('punkt')
nltk.download('averaged_perceptron_tagger')

[nltk_data] Downloading package punkt to /home/lalit/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /home/lalit/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!


True

In [2]:
import os
import getpass

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")

We'll also want to set a project name to make things easier for ourselves.

In [3]:
from uuid import uuid4

os.environ["LANGCHAIN_PROJECT"] = f"AIM - SDG - {uuid4().hex[0:8]}"

OpenAI's API Key!

In [4]:
os.environ["OPENAI_API_KEY"] = getpass.getpass("OpenAI API Key:")

## Generating Synthetic Test Data

We wil be using Ragas to build out a set of synthetic test questions, references, and reference contexts. This is useful because it will allow us to find out how our system is performing.

> NOTE: Ragas is best suited for finding *directional* changes in your LLM-based systems. The absolute scores aren't comparable in a vacuum.

### Data Preparation

We'll prepare our data - which should hopefull be familiar at this point since it's our Use-Case Data!

Next, let's load our data into a familiar LangChain format using the `DirectoryLoader`.

In [5]:
from langchain_community.document_loaders import DirectoryLoader
from langchain_community.document_loaders import PyMuPDFLoader


path = "data/"
loader = DirectoryLoader(path, glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

### Knowledge Graph Based Synthetic Generation

Ragas uses a knowledge graph based approach to create data. This is extremely useful as it allows us to create complex queries rather simply. The additional testset complexity allows us to evaluate larger problems more effectively, as systems tend to be very strong on simple evaluation tasks.

Let's start by defining our `generator_llm` (which will generate our questions, summaries, and more), and our `generator_embeddings` which will be useful in building our graph.

### Unrolled SDG

In [6]:
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_embeddings = LangchainEmbeddingsWrapper(OpenAIEmbeddings())

Next, we're going to instantiate our Knowledge Graph.

This graph will contain N number of nodes that have M number of relationships. These nodes and relationships (AKA "edges") will define our knowledge graph and be used later to construct relevant questions and responses.

In [7]:
from ragas.testset.graph import KnowledgeGraph

kg = KnowledgeGraph()
kg

KnowledgeGraph(nodes: 0, relationships: 0)

The first step we're going to take is to simply insert each of our full documents into the graph. This will provide a base that we can apply transformations to.

In [8]:
from ragas.testset.graph import Node, NodeType

### NOTICE: We're using a subset of the data for this example - this is to keep costs/time down.
for doc in docs:
    kg.nodes.append(
        Node(
            type=NodeType.DOCUMENT,
            properties={"page_content": doc.page_content, "document_metadata": doc.metadata}
        )
    )
kg

KnowledgeGraph(nodes: 64, relationships: 0)

Now, we'll apply the *default* transformations to our knowledge graph. This will take the nodes currently on the graph and transform them based on a set of [default transformations](https://docs.ragas.io/en/latest/references/transforms/#ragas.testset.transforms.default_transforms).

These default transformations are dependent on the corpus length, in our case:

- Producing Summaries -> produces summaries of the documents
- Extracting Headlines -> finding the overall headline for the document
- Theme Extractor -> extracts broad themes about the documents

It then uses cosine-similarity and heuristics between the embeddings of the above transformations to construct relationships between the nodes.

In [9]:
from ragas.testset.transforms import default_transforms, apply_transforms

transformer_llm = generator_llm
embedding_model = generator_embeddings

default_transforms = default_transforms(documents=docs, llm=transformer_llm, embedding_model=embedding_model)
apply_transforms(kg, default_transforms)
kg

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node 'e77473'. Skipping!
Property 'summary' already exists in node '692d84'. Skipping!
Property 'summary' already exists in node '7d492d'. Skipping!
Property 'summary' already exists in node '50617b'. Skipping!
Property 'summary' already exists in node 'd2c64c'. Skipping!
Property 'summary' already exists in node '85e274'. Skipping!
Property 'summary' already exists in node '7affe6'. Skipping!
Property 'summary' already exists in node 'ad4c52'. Skipping!
Property 'summary' already exists in node 'f4e5df'. Skipping!
Property 'summary' already exists in node '3b6024'. Skipping!
Property 'summary' already exists in node '8769ba'. Skipping!
Property 'summary' already exists in node '6884e1'. Skipping!
Property 'summary' already exists in node '55a2a9'. Skipping!
Property 'summary' already exists in node '48ac9c'. Skipping!
Property 'summary' already exists in node 'a7d65e'. Skipping!
Property 'summary' already exists in node 'e78141'. Skipping!
Property

Applying CustomNodeFilter:   0%|          | 0/8 [00:00<?, ?it/s]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/48 [00:00<?, ?it/s]

Property 'summary_embedding' already exists in node 'f4e5df'. Skipping!
Property 'summary_embedding' already exists in node '692d84'. Skipping!
Property 'summary_embedding' already exists in node '50617b'. Skipping!
Property 'summary_embedding' already exists in node '85e274'. Skipping!
Property 'summary_embedding' already exists in node '8769ba'. Skipping!
Property 'summary_embedding' already exists in node '55a2a9'. Skipping!
Property 'summary_embedding' already exists in node 'ad4c52'. Skipping!
Property 'summary_embedding' already exists in node '7affe6'. Skipping!
Property 'summary_embedding' already exists in node '48ac9c'. Skipping!
Property 'summary_embedding' already exists in node '7d492d'. Skipping!
Property 'summary_embedding' already exists in node 'd2c64c'. Skipping!
Property 'summary_embedding' already exists in node 'e77473'. Skipping!
Property 'summary_embedding' already exists in node '3b6024'. Skipping!
Property 'summary_embedding' already exists in node '6884e1'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

KnowledgeGraph(nodes: 86, relationships: 711)

We can save and load our knowledge graphs as follows.

In [10]:
kg.save("usecase_data_kg.json")
usecase_data_kg = KnowledgeGraph.load("usecase_data_kg.json")
usecase_data_kg

KnowledgeGraph(nodes: 86, relationships: 711)

In [11]:
# Comment this out
#import ragas
#dir(ragas)

Using our knowledge graph, we can construct a "test set generator" - which will allow us to create queries.

In [12]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=embedding_model, knowledge_graph=usecase_data_kg)

However, we'd like to be able to define the kinds of queries we're generating - which is made simple by Ragas having pre-created a number of different "QuerySynthesizer"s.

Each of these Synthetsizers is going to tackle a separate kind of query which will be generated from a scenario and a persona.

In essence, Ragas will use an LLM to generate a persona of someone who would interact with the data - and then use a scenario to construct a question from that data and persona.

In [13]:
from ragas.testset.synthesizers import default_query_distribution, SingleHopSpecificQuerySynthesizer, MultiHopAbstractQuerySynthesizer, MultiHopSpecificQuerySynthesizer

query_distribution = [
        (SingleHopSpecificQuerySynthesizer(llm=generator_llm), 0.5),
        (MultiHopAbstractQuerySynthesizer(llm=generator_llm), 0.25),
        (MultiHopSpecificQuerySynthesizer(llm=generator_llm), 0.25),
]

#### ❓ Question #1:

What are the three types of query synthesizers doing? Describe each one in simple terms.  

SingleHopSpecificQuerySynthesizer —
Looks in a single source to generate questions.
These are fact-based queries that can be answered from one part of the data.
Example: “What is the capital of India?”

MultiHopAbstractQuerySynthesizer —
Uses multiple sources and focuses on conceptual or reasoning-based questions.
These need the model to combine ideas rather than fetch one fact.
Example: “How is OOP viewed differently across programming languages?”

MultiHopSpecificQuerySynthesizer —
Also uses multiple sources, but the goal is to find specific factual details across them.
Example: “How popular was the Matrix movie in different countries?”


Finally, we can use our `TestSetGenerator` to generate our testset!

In [14]:
#testset = generator.generate(testset_size=10, query_distribution=query_distribution)
#testset.to_pandas()

import time, re
from openai import RateLimitError, APIConnectionError

def safe_generate(generator, testset_size, query_distribution, retries=50):
    for attempt in range(retries):
        try:
            testset = generator.generate(
                testset_size=testset_size,
                query_distribution=query_distribution
            )
            return testset

        except RateLimitError as e:
            raw_msg = str(e)
            print(f"[Attempt {attempt+1}/{retries}] RateLimitError: {raw_msg}")

            wait_match = re.search(r"try again in (\d+)m(\d+)s", raw_msg.lower())
            if wait_match:
                minutes, seconds = map(int, wait_match.groups())
                wait_time = minutes * 60 + seconds
            else:
                wait_time = 60
            print(f"Sleeping for {wait_time}s...\n")
            time.sleep(wait_time)

        except APIConnectionError as e:
            raw_msg = str(e)
            print(f"[Attempt {attempt+1}/{retries}] APIConnectionError: {raw_msg}")
            wait_time = min(5 * (attempt+1), 60)  # exponential-ish backoff up to 60s
            print(f"Retrying in {wait_time}s...\n")
            time.sleep(wait_time)

        except Exception as e:
            print(f"Unexpected error: {e}")
            raise

    raise RuntimeError("safe_generate: 50 retries failed, no testset produced")

# =========================
# ✅ Usage
# =========================

testset = safe_generate(
    generator,
    testset_size=10,
    query_distribution=query_distribution,
    retries=50
)

# Convert to pandas DataFrame
df = testset.to_pandas()
print(df.head())

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/11 [00:00<?, ?it/s]

                                          user_input  \
0  What significance does November 2022 hold in r...   
1                 How is ChatGPT used in work tasks?   
2  What does SOC2 code 15 refer to in the context...   
3  What is the significance of July 2025 in the c...   
4  Based on the data showing that non-work messag...   

                                  reference_contexts  \
0  [Introduction ChatGPT launched in November 202...   
1  [Table 1: ChatGPT daily message counts (millio...   
2  [Variation by Occupation Figure 23 presents va...   
3  [Conclusion This paper studies the rapid growt...   
4  [<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...   

                                           reference  \
0    Introduction ChatGPT launched in November 2022.   
1  ChatGPT is most commonly used at work for writ...   
2  The provided context does not specify the mean...   
3  By July 2025, ChatGPT had been used weekly by ...   
4  The increase in non-work message volume, wh

### Abstracted SDG

The above method is the full process - but we can shortcut that using the provided abstractions!

This will generate our knowledge graph under the hood, and will - from there - generate our personas and scenarios to construct our queries.



In [15]:
from ragas.testset import TestsetGenerator

generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_embeddings)
dataset = generator.generate_with_langchain_docs(docs, testset_size=10)

Applying HeadlinesExtractor:   0%|          | 0/21 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/64 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to apply transformation: 'headlines' property not found in this node
unable to ap

Applying SummaryExtractor:   0%|          | 0/38 [00:00<?, ?it/s]

Property 'summary' already exists in node '819ab7'. Skipping!
Property 'summary' already exists in node '5cd7d8'. Skipping!
Property 'summary' already exists in node '282e30'. Skipping!
Property 'summary' already exists in node 'd97cbb'. Skipping!
Property 'summary' already exists in node 'de702b'. Skipping!
Property 'summary' already exists in node '09d79b'. Skipping!
Property 'summary' already exists in node '146510'. Skipping!
Property 'summary' already exists in node '7b83f2'. Skipping!
Property 'summary' already exists in node '261cbc'. Skipping!
Property 'summary' already exists in node 'c521a1'. Skipping!
Property 'summary' already exists in node '901fb4'. Skipping!
Property 'summary' already exists in node '0627cc'. Skipping!
Property 'summary' already exists in node 'd0f384'. Skipping!
Property 'summary' already exists in node 'bda3e0'. Skipping!
Property 'summary' already exists in node '515a28'. Skipping!
Property 'summary' already exists in node '8dec94'. Skipping!
Property

...

Property 'summary_embedding' already exists in node '261cbc'. Skipping!
Property 'summary_embedding' already exists in node '7b83f2'. Skipping!
Property 'summary_embedding' already exists in node '282e30'. Skipping!
Property 'summary_embedding' already exists in node 'd97cbb'. Skipping!
Property 'summary_embedding' already exists in node '5cd7d8'. Skipping!
Property 'summary_embedding' already exists in node '819ab7'. Skipping!
Property 'summary_embedding' already exists in node 'c521a1'. Skipping!
Property 'summary_embedding' already exists in node 'de702b'. Skipping!
Property 'summary_embedding' already exists in node '901fb4'. Skipping!
Property 'summary_embedding' already exists in node '0627cc'. Skipping!
Property 'summary_embedding' already exists in node '146510'. Skipping!
Property 'summary_embedding' already exists in node '09d79b'. Skipping!
Property 'summary_embedding' already exists in node 'd0f384'. Skipping!
Property 'summary_embedding' already exists in node '515a28'. Sk

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Samples:   0%|          | 0/12 [00:00<?, ?it/s]

In [16]:
dataset.to_pandas()

,user_input,reference_contexts,reference,synthesizer_name
0,Korinek and Suh 2024 what they say about AI?,[Introduction ChatGPT launched in November 202...,"The context mentions Korinek and Suh, 2024, in...",single_hop_specifc_query_synthesizer
1,How is ChatGPT used in practical guidance acco...,[Table 1: ChatGPT daily message counts (millio...,Nearly 80% of all ChatGPT usage falls into thr...,single_hop_specifc_query_synthesizer
2,Other Professional what does it mean in ChatGP...,[Variation by Occupation Figure 23 presents va...,Variation by Occupation Figure 23 reports resu...,single_hop_specifc_query_synthesizer
3,How does Writing relate to the overall use of ...,[Conclusion This paper studies the rapid growt...,"Writing is by far the most common work use, ac...",single_hop_specifc_query_synthesizer
4,How do the large language models (LLMs) like C...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,"ChatGPT, launched in November 2022 and based o...",multi_hop_abstract_query_synthesizer
5,How does the message volume comparison between...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,"The context shows that in June 2024, total mes...",multi_hop_abstract_query_synthesizer
6,How does the increase in non-work message data...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that non-work messages increase...,multi_hop_abstract_query_synthesizer
7,How does the increase in non-work mesages (M) ...,[<1-hop>\n\nMonth Non-Work (M) (%) Work (M) (%...,The data shows that non-work messages increase...,multi_hop_abstract_query_synthesizer
8,how do 700 million users use chatgpt for data ...,[<1-hop>\n\nIntroduction ChatGPT launched in N...,The provided context does not explicitly detai...,multi_hop_specific_query_synthesizer
9,"How do the 700 million users of ChatGPT, as of...",[<1-hop>\n\nIntroduction ChatGPT launched in N...,"By July 2025, ChatGPT had been used weekly by ...",multi_hop_specific_query_synthesizer


We'll need to provide our LangSmith API key, and set tracing to "true".

# 🤝 BREAKOUT ROOM #2

## Task 4: LangSmith Dataset

Now we can move on to creating a dataset for LangSmith!

First, we'll need to create a dataset on LangSmith using the `Client`!

We'll name our Dataset to make it easy to work with later.

In [16]:
from langsmith import Client

client = Client()

dataset_name = "Use Case Synthetic Data - AIE8"

langsmith_dataset = client.create_dataset(
    dataset_name=dataset_name,
    description="Synthetic Data for Use Cases"
)

We'll iterate through the RAGAS created dataframe - and add each example to our created dataset!

> NOTE: We need to conform the outputs to the expected format - which in this case is: `question` and `answer`.

In [17]:
for data_row in dataset.to_pandas().iterrows():
  client.create_example(
      inputs={
          "question": data_row[1]["user_input"]
      },
      outputs={
          "answer": data_row[1]["reference"]
      },
      metadata={
          "context": data_row[1]["reference_contexts"]
      },
      dataset_id=langsmith_dataset.id
  )

## Basic RAG Chain

Time for some RAG!


In [18]:
rag_documents = docs

To keep things simple, we'll just use LangChain's recursive character text splitter!


In [19]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 500,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

We'll create our vectorstore using OpenAI's [`text-embedding-3-small`](https://platform.openai.com/docs/guides/embeddings/embedding-models) embedding model.

In [20]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

As usual, we will power our RAG application with Qdrant!

In [21]:
from langchain_community.vectorstores import Qdrant

vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG"
)

In [22]:
retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

To get the "A" in RAG, we'll provide a prompt.

In [23]:
from langchain.prompts import ChatPromptTemplate

RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Context: {context}
Question: {question}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_PROMPT)

As is usual: We'll be using `gpt-4.1-mini` for our RAG!

In [24]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4.1-mini")

Finally, we can set-up our RAG LCEL chain!

In [25]:
from operator import itemgetter
from langchain_core.runnables import RunnablePassthrough, RunnableParallel
from langchain.schema import StrOutputParser

rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | rag_prompt | llm | StrOutputParser()
)

In [26]:
rag_chain.invoke({"question" : "What are people doing with AI these days?"})

'Based on the provided context, people are using AI, particularly generative AI like ChatGPT, in a variety of ways both at work and outside of work. AI is performing workplace tasks by augmenting or automating human labor. People use AI to produce writing, software code, spreadsheets, and other digital products, which distinguishes generative AI from traditional web search engines. Users interact with AI for purposes classified under intents such as Asking (seeking information or advice), Doing (producing output or performing tasks), or Expressing (self-expression or reflection). Additionally, AI serves as co-workers producing output or as co-pilots giving advice to improve human problem-solving and productivity. The range of uses includes occupational activities as well as personal uses related to relationships, personal reflection, gaming, and role play. \n\nIn summary, people are using AI to augment productivity, automate tasks, create digital content, seek information and advice, a

## LangSmith Evaluation Set-up

We'll use OpenAI's GPT-4.1 as our evaluation LLM for our base Evaluators.

In [27]:
eval_llm = ChatOpenAI(model="gpt-4.1")

We'll be using a number of evaluators - from LangSmith provided evaluators, to a few custom evaluators!

In [28]:
from langsmith.evaluation import LangChainStringEvaluator, evaluate

qa_evaluator = LangChainStringEvaluator("qa", config={"llm" : eval_llm})

labeled_helpfulness_evaluator = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {
            "helpfulness": (
                "Is this submission helpful to the user,"
                " taking into account the correct reference answer?"
            )
        },
        "llm" : eval_llm
    },
    prepare_data=lambda run, example: {
        "prediction": run.outputs["output"],
        "reference": example.outputs["answer"],
        "input": example.inputs["question"],
    }
)

dopeness_evaluator = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {
            "dopeness": "Is this response dope, lit, cool, or is it just a generic response?",
        },
        "llm" : eval_llm
    }
)

#### 🏗️ Activity #2:

Highlight what each evaluator is evaluating.

- `qa_evaluator`: 
 Evaluates whether the model’s output matches the reference answer. In other words, is the response factually correct and accurate? 
- `labeled_helpfulness_evaluator`:  
Evaluates if the response was useful and relevant to the user’s question.
- `dopeness_evaluator`:  
Evaluates if the response is engaging or cool, instead of being generic.


## LangSmith Evaluation

In [29]:
evaluate(
    rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "default_chain_init"},
)

View the evaluation results for experiment: 'artistic-candy-55' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/e32834bb-5c95-4db4-898c-8502f5b5d89e/compare?selectedSessions=0b2c8266-b7e3-4c1a-bc7d-f165063a277e




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does the total number of messages sent by ...,Based on the provided context:\n\n- By July 20...,None,"According to the context, by July 2025, ChatGP...",1,1,0,4.662036,a715ec2b-c232-4349-8702-bb5e3d9f6c1e,e96429bc-c130-434a-85f8-a39237528b9e
1,What is the significance of July 2025 in the c...,"By July 2025, ChatGPT had experienced rapid gr...",None,"By July 2025, ChatGPT had become widely adopte...",1,1,0,3.308139,c529cae5-0295-4ed6-a396-2c0ff15d9673,e75cabf6-8da8-44da-aedd-5e968b8214a2
2,How does the fact that 700 million users utili...,The fact that 700 million users utilize ChatGP...,None,"The first segment states that by July 2025, 70...",1,1,0,3.483743,41c9bfda-9dc1-4c14-9a1d-5ccb5f4383fc,197cbb34-5447-4e6f-aa47-db99357e84c6
3,how many users r there with 700 million users ...,"By July 2025, ChatGPT had more than 700 millio...",None,"There are 700 million users of ChatGPT, and by...",1,1,0,3.815441,ac97d192-a8ac-4ec8-a8f1-293a8b78e17a,fd0605b0-6c30-4f73-9265-d6c0360583a7
4,Based on the increasing percentage of non-work...,"Based on the provided context, the percentage ...",None,The data shows that non-work messages have gro...,1,1,0,7.238380,0723e84c-1909-4ccf-9fd7-763496c2aabc,d18d34ed-4ec2-4c1b-acae-194378efe858
5,how chatgpt help work and info like in paper a...,"Based on the provided context, ChatGPT helps w...",None,the paper says chatgpt is used for work and in...,1,1,0,7.537013,7c5d013c-04f0-4eae-bd56-fd751216a67d,a7339946-b36a-4ab7-b940-e6fc262f1bc0
6,hw chatgpt use by work activites and messege d...,Based on the provided context:\n\nChatGPT usag...,None,Variation by Occupation Figure 23 shows that u...,1,1,0,5.931343,98990e5f-cab0-474a-b103-88f1dd383ff4,4db7b13f-58e7-4baf-938a-5fdabbac1cc1
7,how message classification like asking doing e...,"Message classification into Asking, Doing, and...",None,The context explains that ChatGPT is used for ...,1,1,0,4.166061,60a8558d-560f-4330-95df-c770dc37c031,abe3b533-983e-4ba2-be9f-bbb856832136
8,How is ChatGPT being utilized by career-orient...,"According to the provided context, career-orie...",None,"According to the provided context, ChatGPT is ...",0,0,0,2.719820,c5b85346-45d7-4860-aae6-2463bcfce091,493d863c-2227-4039-b745-f7f00502b5ed
9,What is enginering and sciense?,"Based on the provided context, ""Engineering"" a...",None,Variation by Occupation Figure 23 presents var...,1,1,0,6.937622,5c1dd8ea-14bc-4d5f-babc-e98553ec8c84,f1feab67-7f9b-450a-9e8a-2f725f3d5e2a


## Dope-ifying Our Application

We'll be making a few changes to our RAG chain to increase its performance on our SDG evaluation test dataset!

- Include a "dope" prompt augmentation
- Use larger chunks
- Improve the retriever model to: `text-embedding-3-large`

Let's see how this changes our evaluation!

In [30]:
DOPENESS_RAG_PROMPT = """\
Given a provided context and question, you must answer the question based only on context.

If you cannot answer the question based on the context - you must say "I don't know".

Make your answer rad, ensure high levels of dopeness. Do not be generic, or give generic responses.

Context: {context}
Question: {question}
"""

dopeness_rag_prompt = ChatPromptTemplate.from_template(DOPENESS_RAG_PROMPT)

In [31]:
rag_documents = docs

In [32]:
from langchain.text_splitter import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 50
)

rag_documents = text_splitter.split_documents(rag_documents)

#### ❓Question #2:

Why would modifying our chunk size modify the performance of our application?  

Changing chunk size changes how much context is inside each embedding.
Smaller chunks = more precise but may miss connections.
Larger chunks = broader context but can add noise.  

Chunk size → like how big you cut the cake slices.
If slices are too small, you get crumbs (tiny context).
If slices are too big, you mix flavors (irrelevant info).
The right slice size = balanced, clear bites of meaning.


In [33]:
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

#### ❓Question #3:

Why would modifying our embedding model modify the performance of our application?  

Embedding model quality decides how well text meaning is captured.
Stronger embeddings = better similarity matches and richer understanding.=
Weaker ones = shallow or wrong matches, hurting accuracy.  

Suppose your embedding model is small or trained mostly on surface-level co-occurrence (not deep meaning).  
It might say these two words are “similar”:  
“bank” and “river”

Why?
Because it sees them often appear together, “river bank,” “bank of the river.”  
But the intended meaning might be financial (“bank account”), not geographical.

A better embedding model (like text-embedding-3-large) understands context and separates the meanings:
“bank” (finance) is closer to “money” or “loan.”
“bank” (geography) is closer to “shore” or “stream.”

✅ So a weak model confuses word proximity with semantic similarity, while a strong one understands contextual meaning.  





Analogy  

Embedding model:  like how sharp your knife is.
A sharper knife (better embedding model) cuts more precisely, capturing the true shape of the text’s meaning.
A dull knife (weaker model) makes rough, blurry cuts words that shouldn’t be connected end up stuck together.
Together (with Chunking strategy): The right slice size (Chunking strategy) + the right knife sharpness (Embedding) = clean, meaningful chunks that the retriever can actually understand

In [34]:
vectorstore = Qdrant.from_documents(
    documents=rag_documents,
    embedding=embeddings,
    location=":memory:",
    collection_name="Use Case RAG Docs"
)

In [35]:
retriever = vectorstore.as_retriever()

Setting up our new and improved DOPE RAG CHAIN.

In [36]:
dopeness_rag_chain = (
    {"context": itemgetter("question") | retriever, "question": itemgetter("question")}
    | dopeness_rag_prompt | llm | StrOutputParser()
)

Let's test it on the same output that we saw before.

In [37]:
dopeness_rag_chain.invoke({"question" : "How are people using AI to make money?"})

'Yo, here’s the scoop straight from the digital trenches: People aren’t just handing over tasks to AI like some soulless robot slaves—they’re teaming up with ChatGPT as a slick advisor and brainiac research sidekick. The AI boosts their hustle by cranking decision-making to beast mode, especially in those knowledge-heavy gigs where every smart move stacks up the cash. Collis and Brynjolfsson (2025) even dropped a bombshell that US users value generative AI so much they’d need a cool $98 to skip it for a month—translating to a mind-blowing $97 billion boost a year. So it’s not just AI grinding away; it’s amplifying human smarts to cash in smarter, not just harder. Total boss move in the money-making game.'

Finally, we can evaluate the new chain on the same test set!

In [38]:
evaluate(
    dopeness_rag_chain.invoke,
    data=dataset_name,
    evaluators=[
        qa_evaluator,
        labeled_helpfulness_evaluator,
        dopeness_evaluator
    ],
    metadata={"revision_id": "dopeness_rag_chain"},
)

View the evaluation results for experiment: 'tart-prose-81' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/e32834bb-5c95-4db4-898c-8502f5b5d89e/compare?selectedSessions=e196d5d4-0b81-4b6e-91c2-08e78e5856cc




0it [00:00, ?it/s]

,inputs.question,outputs.output,error,reference.answer,feedback.correctness,feedback.helpfulness,feedback.dopeness,execution_time,example_id,id
0,How does the total number of messages sent by ...,"Alright, here’s the ultra-doper breakdown base...",None,"According to the context, by July 2025, ChatGP...",0,0,1,7.811460,a715ec2b-c232-4349-8702-bb5e3d9f6c1e,4784f5d5-150d-4623-9fbf-0958c33121c1
1,What is the significance of July 2025 in the c...,"Ah, July 2025 marks a legendary milestone in t...",None,"By July 2025, ChatGPT had become widely adopte...",1,1,1,5.862221,c529cae5-0295-4ed6-a396-2c0ff15d9673,7e82ce10-e546-4a5d-afc4-1c6514b9d4df
2,How does the fact that 700 million users utili...,"Oh, buckle up — here’s the dopest scoop straig...",None,"The first segment states that by July 2025, 70...",1,1,1,6.404322,41c9bfda-9dc1-4c14-9a1d-5ccb5f4383fc,577f731f-be9f-46e2-8619-82d457f62f61
3,how many users r there with 700 million users ...,"Yo, buckle up because these numbers are absolu...",None,"There are 700 million users of ChatGPT, and by...",1,1,1,5.047822,ac97d192-a8ac-4ec8-a8f1-293a8b78e17a,07032e1b-009a-4922-8ffe-1272bc8511de
4,Based on the increasing percentage of non-work...,"Alright, let’s dive into this AI wave like a b...",None,The data shows that non-work messages have gro...,1,1,1,7.672431,0723e84c-1909-4ccf-9fd7-763496c2aabc,422caa4d-3212-4c8d-aee6-7e2f0b665450
5,how chatgpt help work and info like in paper a...,"Alright, buckle up for the ultimate breakdown ...",None,the paper says chatgpt is used for work and in...,1,1,1,9.875988,7c5d013c-04f0-4eae-bd56-fd751216a67d,5aced158-b774-4bfd-a212-6de2c33ca101
6,hw chatgpt use by work activites and messege d...,"Yo, buckle up for this turbo-charged deep dive...",None,Variation by Occupation Figure 23 shows that u...,1,1,1,8.436108,98990e5f-cab0-474a-b103-88f1dd383ff4,7388eaef-d78a-4898-90b4-acabd53b0599
7,how message classification like asking doing e...,"Alright, buckle up for a turbocharged insight ...",None,The context explains that ChatGPT is used for ...,1,0,1,7.069418,60a8558d-560f-4330-95df-c770dc37c031,28694512-6427-444f-bcc8-399bc0b6165c
8,How is ChatGPT being utilized by career-orient...,"Alright, buckle up — here’s how ChatGPT is *su...",None,"According to the provided context, ChatGPT is ...",1,1,1,4.614060,c5b85346-45d7-4860-aae6-2463bcfce091,8af1a0b2-83e4-4b49-ad11-ec9070220957
9,What is enginering and sciense?,"Yo, let’s break down the vibes of Engineering ...",None,Variation by Occupation Figure 23 presents var...,0,0,1,7.221623,5c1dd8ea-14bc-4d5f-babc-e98553ec8c84,519a82be-1c58-4e88-bbcc-d506ae627094


#### 🏗️ Activity #3:

Provide a screenshot of the difference between the two chains, and explain why you believe certain metrics changed in certain ways.  


Screenshot 1 : 
![Evaluation Results](/images/LangSmith_Eval_Before_Dopeness.png)



Screenshot 2 : After adding Dopeness  
![Evaluation Results](/images/LangSmith_Eval_After_Dopeness.png)
